In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb

In [2]:
# Load all CSV files
telemetry = pd.read_csv('PdM_telemetry.csv')
errors = pd.read_csv('PdM_errors.csv') 
failures = pd.read_csv('PdM_failures.csv')
maintenance = pd.read_csv('PdM_maint.csv')
machines = pd.read_csv('PdM_machines.csv')

print(f"Telemetry shape: {telemetry.shape}")
print(f"Failures shape: {failures.shape}")

Telemetry shape: (876100, 6)
Failures shape: (761, 3)


In [3]:
# Convert datetime columns
telemetry['datetime'] = pd.to_datetime(telemetry['datetime'])
failures['datetime'] = pd.to_datetime(failures['datetime'])

In [4]:
telemetry_with_labels = telemetry.copy()
telemetry_with_labels['will_fail_24h'] = 0

# For each failure, mark previous 24 hours as positive
for idx, failure_row in failures.iterrows():
    machine_id = failure_row['machineID']
    failure_time = failure_row['datetime']
    
    # Find telemetry records 24 hours before this failure
    start_time = failure_time - pd.Timedelta(hours=24)
    
    # Mark these records as positive (will fail)
    mask = ((telemetry_with_labels['machineID'] == machine_id) & 
            (telemetry_with_labels['datetime'] >= start_time) & 
            (telemetry_with_labels['datetime'] < failure_time))
    
    telemetry_with_labels.loc[mask, 'will_fail_24h'] = 1

print(f"Positive samples (will fail): {telemetry_with_labels['will_fail_24h'].sum()}")
print(f"Negative samples (won't fail): {len(telemetry_with_labels) - telemetry_with_labels['will_fail_24h'].sum()}")

Positive samples (will fail): 17184
Negative samples (won't fail): 858916


In [5]:
# Sort by machine and time for rolling calculations
telemetry_with_labels = telemetry_with_labels.sort_values(['machineID', 'datetime'])

# Create rolling window features (last 6 hours averages and trends)
window_size = 6
feature_data = []

for machine_id in telemetry_with_labels['machineID'].unique():
    machine_data = telemetry_with_labels[telemetry_with_labels['machineID'] == machine_id].copy()
    
    # Rolling averages
    machine_data['volt_avg_6h'] = machine_data['volt'].rolling(window_size).mean()
    machine_data['rotate_avg_6h'] = machine_data['rotate'].rolling(window_size).mean()
    machine_data['pressure_avg_6h'] = machine_data['pressure'].rolling(window_size).mean()
    machine_data['vibration_avg_6h'] = machine_data['vibration'].rolling(window_size).mean()
    
    # Rolling standard deviations (variability)
    machine_data['volt_std_6h'] = machine_data['volt'].rolling(window_size).std()
    machine_data['rotate_std_6h'] = machine_data['rotate'].rolling(window_size).std()
    machine_data['pressure_std_6h'] = machine_data['pressure'].rolling(window_size).std()
    machine_data['vibration_std_6h'] = machine_data['vibration'].rolling(window_size).std()
    
    # Trends (current value - average)
    machine_data['volt_trend'] = machine_data['volt'] - machine_data['volt_avg_6h']
    machine_data['rotate_trend'] = machine_data['rotate'] - machine_data['rotate_avg_6h']
    machine_data['pressure_trend'] = machine_data['pressure'] - machine_data['pressure_avg_6h']
    machine_data['vibration_trend'] = machine_data['vibration'] - machine_data['vibration_avg_6h']
    
    feature_data.append(machine_data)


In [6]:
full_data = pd.concat(feature_data, ignore_index=True)

In [7]:
# Add error count features
error_counts = errors.groupby(['machineID', 'datetime']).size().reset_index(name='error_count')
error_counts['datetime'] = pd.to_datetime(error_counts['datetime'])

# Merge error counts
full_data = full_data.merge(error_counts, on=['machineID', 'datetime'], how='left')
full_data['error_count'] = full_data['error_count'].fillna(0)

In [8]:
full_data = full_data.merge(machines, on='machineID', how='left')

In [9]:
full_data.head()

,datetime,machineID,volt,rotate,pressure,vibration,will_fail_24h,volt_avg_6h,rotate_avg_6h,pressure_avg_6h,...,rotate_std_6h,pressure_std_6h,vibration_std_6h,volt_trend,rotate_trend,pressure_trend,vibration_trend,error_count,model,age
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,model3,18
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,model3,18
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,model3,18
3,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,model3,18
4,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,model3,18


In [10]:
full_data = full_data.dropna()

In [11]:
full_data.head()

,datetime,machineID,volt,rotate,pressure,vibration,will_fail_24h,volt_avg_6h,rotate_avg_6h,pressure_avg_6h,...,rotate_std_6h,pressure_std_6h,vibration_std_6h,volt_trend,rotate_trend,pressure_trend,vibration_trend,error_count,model,age
5,2015-01-01 11:00:00,1,172.504839,430.323362,95.927042,35.655017,0,167.110779,426.741827,100.139769,...,58.903479,14.478264,7.106854,5.394061,3.581535,-4.212728,-1.919679,0.0,model3,18
6,2015-01-01 12:00:00,1,156.556031,499.071623,111.755684,42.753920,0,163.833808,440.169751,99.919394,...,65.467524,14.250211,6.663459,-7.277778,58.901872,11.836290,5.568184,0.0,model3,18
7,2015-01-01 13:00:00,1,172.522781,409.624717,101.001083,35.482009,0,165.441068,441.315956,100.842821,...,64.737430,14.082009,5.926790,7.081713,-31.691239,0.158262,-0.381733,0.0,model3,18
8,2015-01-01 14:00:00,1,175.324524,398.648781,110.624361,45.482287,0,166.163505,419.865782,106.740563,...,50.224845,6.676548,6.985945,9.161019,-21.217001,3.883797,7.734639,0.0,model3,18
9,2015-01-01 15:00:00,1,169.218423,460.850670,104.848230,39.901735,0,167.289436,438.982671,106.007175,...,36.511886,6.587035,6.885082,1.928987,21.867999,-1.158945,2.357489,0.0,model3,18


In [12]:
print(f"Telemetry shape: {full_data.shape}")

Telemetry shape: (875600, 22)


In [13]:
# Select features for training
feature_columns = [
    'volt', 'rotate', 'pressure', 'vibration',  # Current readings
    'volt_avg_6h', 'rotate_avg_6h', 'pressure_avg_6h', 'vibration_avg_6h',  # Averages
    'volt_std_6h', 'rotate_std_6h', 'pressure_std_6h', 'vibration_std_6h',  # Variability
    'volt_trend', 'rotate_trend', 'pressure_trend', 'vibration_trend',  # Trends
    'error_count',  # Error information
    'age', 'model'  # Machine characteristics
]

# Handle categorical variables (model type)
full_data['model'] = full_data['model'].astype('category').cat.codes

# Prepare X (features) and y (target)
X = full_data[feature_columns]
y = full_data['will_fail_24h']

print(f"Training data shape: X={X.shape}, y={y.shape}")
print(f"Class distribution: {y.value_counts()}")

Training data shape: X=(875600, 19), y=(875600,)
Class distribution: will_fail_24h
0    858511
1     17089
Name: count, dtype: int64


In [14]:
# Split into train and test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

Training set: (700480, 19)
Test set: (175120, 19)


In [15]:
# Handle class imbalance
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
print(f"Scale pos weight (for imbalance): {scale_pos_weight:.2f}")

Scale pos weight (for imbalance): 50.24


In [16]:
# Create XGBoost model
model = xgb.XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=scale_pos_weight,  # Handle imbalanced data
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [17]:
# Predict on test set
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Get probabilities
y_pred = model.predict(X_test)  # Get binary predictions (0 or 1)

In [18]:
# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.90      0.94    171702
           1       0.15      0.95      0.26      3418

    accuracy                           0.90    175120
   macro avg       0.58      0.92      0.60    175120
weighted avg       0.98      0.90      0.93    175120



In [37]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
# Calculate detailed metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f"\n📊 DETAILED PERFORMANCE METRICS:")
print(f"Accuracy:  {accuracy:.4f} ({accuracy:.1%})")
print(f"Precision: {precision:.4f} ({precision:.1%})")
print(f"Recall:    {recall:.4f} ({recall:.1%})")
print(f"F1-Score:  {f1:.4f} ({f1:.1%})")
print(f"AUC Score: {auc_score:.4f} ({auc_score:.1%})")


📊 DETAILED PERFORMANCE METRICS:
Accuracy:  0.8969 (89.7%)
Precision: 0.1535 (15.4%)
Recall:    0.9485 (94.9%)
F1-Score:  0.2643 (26.4%)
AUC Score: 0.9612 (96.1%)


In [20]:
# Print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:
[[153826  17876]
 [   176   3242]]


In [21]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))


Top 10 Most Important Features:
             feature  importance
6    pressure_avg_6h    0.259464
5      rotate_avg_6h    0.192253
4        volt_avg_6h    0.191518
7   vibration_avg_6h    0.175501
18             model    0.071774
17               age    0.030955
16       error_count    0.010943
8        volt_std_6h    0.007303
10   pressure_std_6h    0.006768
3          vibration    0.006034


In [22]:
# Show some example predictions
example_indices = [0, 100, 200, 300, 400]
print("\nExample Predictions:")
print("Index | Actual | Predicted | Probability | Risk Level")
print("-" * 55)

for idx in example_indices:
    if idx < len(y_test):
        actual = y_test.iloc[idx]
        predicted = y_pred[idx]
        probability = y_pred_proba[idx]
        
        risk_level = "HIGH" if probability > 0.5 else "LOW"
        
        print(f"{idx:5d} | {actual:6d} | {predicted:9d} | {probability:11.3f} | {risk_level}")



Example Predictions:
Index | Actual | Predicted | Probability | Risk Level
-------------------------------------------------------
    0 |      0 |         0 |       0.040 | LOW
  100 |      0 |         0 |       0.027 | LOW
  200 |      0 |         1 |       0.649 | HIGH
  300 |      0 |         1 |       0.893 | HIGH
  400 |      0 |         0 |       0.201 | LOW


In [24]:
# Save model for future use
model.save_model('failure_prediction_model.json')

In [25]:
def predict_machine_failure(machine_id, target_datetime):
    """
    Predict if a machine will fail in the next 24 hours
    
    Parameters:
    - machine_id: ID of the machine to check
    - target_datetime: datetime to make prediction from (format: 'YYYY-MM-DD HH:MM:SS')
    
    Returns:
    - Dictionary with prediction results
    """
    
    # Convert input datetime
    target_time = pd.to_datetime(target_datetime)
    
    # Check if machine exists
    if machine_id not in full_data['machineID'].unique():
        return {"error": f"Machine ID {machine_id} not found in dataset"}
    
    # Get machine data up to target datetime
    machine_history = full_data[
        (full_data['machineID'] == machine_id) & 
        (full_data['datetime'] <= target_time)
    ].sort_values('datetime')
    
    if len(machine_history) == 0:
        return {"error": f"No data found for Machine {machine_id} at {target_datetime}"}
    
    # Get the latest available record
    latest_record = machine_history.iloc[-1]
    
    # Prepare features for prediction
    prediction_features = []
    for col in feature_columns:
        if col in latest_record:
            prediction_features.append(latest_record[col])
        else:
            prediction_features.append(0)  # Default value if missing
    
    # Reshape for prediction (model expects 2D array)
    features_array = np.array(prediction_features).reshape(1, -1)
    
    # Make prediction
    failure_probability = model.predict_proba(features_array)[0, 1]
    will_fail = failure_probability > 0.5
    
    # Get current sensor readings for display
    current_sensors = {
        'voltage': latest_record['volt'],
        'rotation': latest_record['rotate'], 
        'pressure': latest_record['pressure'],
        'vibration': latest_record['vibration']
    }
    
    # Return results
    result = {
        "machine_id": machine_id,
        "prediction_time": target_datetime,
        "data_time": latest_record['datetime'].strftime('%Y-%m-%d %H:%M:%S'),
        "will_fail_24h": will_fail,
        "failure_probability": round(failure_probability, 3),
        "risk_level": "HIGH" if failure_probability > 0.7 else "MEDIUM" if failure_probability > 0.3 else "LOW",
        "current_sensors": current_sensors,
        "recommendation": "Schedule immediate maintenance" if will_fail else "Continue normal operation"
    }
    
    return result


In [31]:
# Test with different machines and times
test_cases = [
    {"machine_id": 1, "datetime": "2015-01-04 06:00:00"},
    {"machine_id": 5, "datetime": "2015-08-20 08:00:00"}, 
    {"machine_id": 10, "datetime": "2015-10-01 14:00:00"},
    {"machine_id": 25, "datetime": "2015-12-15 10:00:00"}
]
for i, test_case in enumerate(test_cases, 1):
    print(f"\n--- TEST CASE {i} ---")
    result = predict_machine_failure(test_case["machine_id"], test_case["datetime"])
    
    if "error" in result:
        print(f"❌ {result['error']}")
    else:
        print(f"🏭 Machine ID: {result['machine_id']}")
        print(f"📅 Prediction Time: {result['prediction_time']}")
        print(f"📊 Latest Data From: {result['data_time']}")
        print(f"⚠️  Will Fail in 24h: {'YES' if result['will_fail_24h'] else 'NO'}")
        print(f"📈 Failure Probability: {result['failure_probability']:.1%}")
        print(f"🚨 Risk Level: {result['risk_level']}")
        print(f"💡 Recommendation: {result['recommendation']}")
        print(f"📡 Current Sensors:")
        for sensor, value in result['current_sensors'].items():
            print(f"   - {sensor.capitalize()}: {value:.2f}")



--- TEST CASE 1 ---
🏭 Machine ID: 1
📅 Prediction Time: 2015-01-04 06:00:00
📊 Latest Data From: 2015-01-04 06:00:00
⚠️  Will Fail in 24h: YES
📈 Failure Probability: 99.0%
🚨 Risk Level: HIGH
💡 Recommendation: Schedule immediate maintenance
📡 Current Sensors:
   - Voltage: 165.01
   - Rotation: 448.47
   - Pressure: 97.71
   - Vibration: 48.24

--- TEST CASE 2 ---
🏭 Machine ID: 5
📅 Prediction Time: 2015-08-20 08:00:00
📊 Latest Data From: 2015-08-20 08:00:00
⚠️  Will Fail in 24h: NO
📈 Failure Probability: 0.3%
🚨 Risk Level: LOW
💡 Recommendation: Continue normal operation
📡 Current Sensors:
   - Voltage: 168.51
   - Rotation: 495.08
   - Pressure: 103.25
   - Vibration: 33.53

--- TEST CASE 3 ---
🏭 Machine ID: 10
📅 Prediction Time: 2015-10-01 14:00:00
📊 Latest Data From: 2015-10-01 14:00:00
⚠️  Will Fail in 24h: NO
📈 Failure Probability: 0.3%
🚨 Risk Level: LOW
💡 Recommendation: Continue normal operation
📡 Current Sensors:
   - Voltage: 180.49
   - Rotation: 472.53
   - Pressure: 95.80
   -

In [27]:
def interactive_prediction():
    """Interactive function to test with custom inputs"""
    
    print("\nEnter machine details for failure prediction:")
    print("Available Machine IDs: 1-100")
    print("Date format: YYYY-MM-DD HH:MM:SS (e.g., 2015-06-15 12:00:00)")
    
    try:
        # Get user input
        machine_id = int(input("\nEnter Machine ID (1-100): "))
        datetime_str = input("Enter datetime (YYYY-MM-DD HH:MM:SS): ")
        
        # Make prediction
        result = predict_machine_failure(machine_id, datetime_str)
        
        print("\n" + "-"*60)
        print("🔮 PREDICTION RESULT")
        print("-"*60)
        
        if "error" in result:
            print(f"❌ Error: {result['error']}")
        else:
            # Display result in a nice format
            status = "⚠️  WILL FAIL" if result['will_fail_24h'] else "✅ SAFE"
            print(f"Machine {result['machine_id']}: {status}")
            print(f"Failure Probability: {result['failure_probability']:.1%}")
            print(f"Risk Level: {result['risk_level']}")
            print(f"Recommendation: {result['recommendation']}")
            
            if result['will_fail_24h']:
                print("🚨 ALERT: Schedule maintenance immediately!")
            else:
                print("👍 Machine is operating normally")
    
    except KeyboardInterrupt:
        print("\nPrediction cancelled.")
    except Exception as e:
        print(f"Error: {e}")


In [32]:
interactive_prediction()


Enter machine details for failure prediction:
Available Machine IDs: 1-100
Date format: YYYY-MM-DD HH:MM:SS (e.g., 2015-06-15 12:00:00)



Enter Machine ID (1-100):  12
Enter datetime (YYYY-MM-DD HH:MM:SS):  2015-06-15 12:00:00



------------------------------------------------------------
🔮 PREDICTION RESULT
------------------------------------------------------------
Machine 12: ✅ SAFE
Failure Probability: 4.1%
Risk Level: LOW
Recommendation: Continue normal operation
👍 Machine is operating normally


In [36]:
interactive_prediction()


Enter machine details for failure prediction:
Available Machine IDs: 1-100
Date format: YYYY-MM-DD HH:MM:SS (e.g., 2015-06-15 12:00:00)



Enter Machine ID (1-100):  1
Enter datetime (YYYY-MM-DD HH:MM:SS):   2015-01-04 08:00:00



------------------------------------------------------------
🔮 PREDICTION RESULT
------------------------------------------------------------
Machine 1: ⚠️  WILL FAIL
Failure Probability: 92.9%
Risk Level: HIGH
Recommendation: Schedule immediate maintenance
🚨 ALERT: Schedule maintenance immediately!
